# المرحلة الأولى: تجهيز بيانات اللهجات العربية

في هذا الدفتر نشرح كيف انتقلنا من الملفات الخام إلى بيانات موحدة قابلة للتدريب.

**المصادر المستخدمة في التدريب:** IADD وMADAR.  
**المصادر التي لم تدخل التدريب:** NADI المتاح TEST فقط، وQADI الذي يعتمد تدريبُه على Twitter IDs.


## أهداف هذه المرحلة

سنقوم بـ:

1. التعرف على بنية السجل الموحد.
2. تطبيق تطبيع النص العربي.
3. توحيد أسماء المناطق والدول.
4. إزالة السجلات المكررة.
5. فحص توزيع البيانات وتقسيماتها.

ملفات Python الأصلية التي تنفذ الدمج الكامل موجودة في `scripts/prepare_data.py`.


In [ ]:
# Path ??????? ?? ?????? ??????? ?????? ???? ??? Windows ?Linux
from pathlib import Path
# json ?????? ????? JSON/JSONL
import json
# pandas ???? ???????? ?????????? ?? ?????
import pandas as pd

# ???? ???? ??????? ??????
ROOT = Path.cwd()
# ???? ???? ???????? ???? ?? ??????? ??????
PROCESSED = ROOT / "data" / "processed"
# ???? ?????? ?????? ?? ?? ?????? ???? ?? ???? ???????
print("???? ???????:", ROOT)
# ???? ??????? ???????? ?? ???? ???????? ????????
print("??????? ????????:", sorted(path.name for path in PROCESSED.glob("*")))

## شكل السجل الموحد

كل نص أصبح مرتبطًا بالحقول التالية:

- `text`: النص المنظف.
- `region`: المنطقة.
- `country`: الدولة.
- `city`: المدينة إن كانت متوفرة.
- `source`: مصدر السجل.


In [ ]:
# ???? ??? ???????? ??????? ??? ????? ?????? ???????
sample_path = PROCESSED / "all.jsonl"
# ???? ????? ?????? UTF-8 ?????? ??? ?????? ???????
with sample_path.open(encoding="utf-8") as handle:
    # ???? ??? ??? ??? ?????? ??? ??? ?????
    first_row = json.loads(next(handle))
# ???? ????? ???? ?????? ???????
first_row

## دالة تطبيع النص

التطبيع المستخدم في المشروع يزيل الروابط والتشكيل والتطويل، ويوحد بعض أشكال الحروف.
لا نستخدم Lemmatization أو Stopword Removal؛ لأن النموذج يعتمد لاحقًا على Character n-grams.


In [ ]:
# re ?????? ????? ????? ????? ?????????? ???? ??????
import re

# ??? ????? ??????? ?????? ??? ????? ?? ????
ARABIC_DIACRITICS = re.compile(r"[\u0610-\u061a\u064b-\u065f\u0670\u06d6-\u06ed]")
# ??? ????? ??????? ???? ?? ?????? ?????? ?? ???????
URL = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)

# ??? ???? ??????? ?? ???? ??????? ???????? ?? prepare_data.py
def normalize_text_demo(value):
    # ???? ?????? ??? ?? ?????? ?????? ??????? ??? ????
    text = str(value or "")
    # ?????? ??????? ?????
    text = URL.sub(" ", text)
    # ???? ??????? ?????? ?
    text = text.replace("\u0640", "")
    # ???? ???????
    text = ARABIC_DIACRITICS.sub("", text)
    # ???? ????? ????? ???????? ??? ?
    text = re.sub("[????]", "?", text)
    # ???? ????? ?????? ???????? ?????? ????????
    text = text.replace("?", "?").replace("?", "?")
    # ???? ???????? ??????? ?? ????? ?????? ????
    return re.sub(r"\s+", " ", text).strip()

# ????? ????? ??????? ??? ???????
examples = ["??????? https://example.com", "????????? ?? ?????"]
# ???? ???? ??? ??????? ?????
[(text, normalize_text_demo(text)) for text in examples]

## قراءة البيانات وحساب الإحصاءات

التقرير المحفوظ يوضح حجم البيانات بعد كل مرحلة وتوزيعها حسب المصدر والمنطقة والدولة.


In [ ]:
# ???? ????? ???????? ???? ????? ????? ????? ????????
report = json.loads((PROCESSED / "dataset_report.json").read_text(encoding="utf-8"))
# ?????? ??? ????? ???????? ?? ????? ????
summary = {
    "??????? ?????": report["raw_rows"],
    "??? ????????": report["after_preprocessing_rows"],
    "??? ????? ???????": report["deduplicated_rows"],
    "???????": report["split_sizes"]["train"],
    "??????": report["split_sizes"]["validation"],
    "????????": report["split_sizes"]["test"],
}
# ???? ??????? ??? Series ????? ?? ???? ????
pd.Series(summary, name="??????")

In [ ]:
# ???? ????? ??????? ??? ??????? ?? ???????
region_distribution = pd.Series(report["overall_distribution"]["region"])
# ???? ??????? ?? ?????? ??? ??????
region_distribution = region_distribution.sort_values(ascending=False)
# ???? ??????? ?? ????
region_distribution.to_frame("??? ??????")

## ملاحظة مهمة

المدن موجودة في بيانات MADAR، لكن هدف النسخة النهائية هو تصنيف المنطقة والدولة.
لذلك لم ندرب نموذج مدينة ولم نعرضها في الواجهة.
